# permute-back-argsort — worked example 2: Permute backward on 4D tensor with arbitrary dims

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `permute-back-argsort`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The argsort-based inverse permutation works for tensors of any dimensionality. For a 4D tensor with `dims = (3, 1, 0, 2)`, `argsort` returns `(2, 1, 3, 0)` — you can verify: position of 0 in `(3,1,0,2)` is 2, position of 1 is 1, position of 2 is 3, position of 3 is 0. Applying this inverse to `grad_out` restores the original axis ordering of `x`.

## Worked solution

**Step 1 — Build a 4D test tensor.** We create `x` with shape `(2, 3, 4, 5)` and apply a non-trivial permutation `dims = (3, 1, 0, 2)`, yielding `y` with shape `(5, 3, 2, 4)`.

**Step 2 — Compute inverse.** `np.argsort([3, 1, 0, 2])` = `[2, 1, 3, 0]` because: 0 is at position 2 in dims, 1 is at position 1, 2 is at position 3, 3 is at position 0.

**Step 3 — Apply inverse to grad_out.** `grad_out.permute(2, 1, 3, 0)` has shape `(2, 3, 4, 5)` — same as `x`. The backward is complete.

**Step 4 — Numerical verification.** We check both shape equality (`grad_x.shape == x.shape`) and value equality (`t.equal(y.permute(*inverse), x)`) to confirm the round-trip property holds for 4D tensors.

In [ ]:
import torch as t
import numpy as np

def permute_back(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor, dims: tuple) -> t.Tensor:
    """Backward of np.transpose / torch.permute via argsort inverse."""
    inverse = tuple(int(i) for i in np.argsort(dims))
    return grad_out.permute(*inverse)

# 4D scenario
t.manual_seed(55)
x = t.randn(2, 3, 4, 5)
dims = (3, 1, 0, 2)

y = x.permute(*dims)
print(f'x shape: {x.shape}')    # (2, 3, 4, 5)
print(f'y shape: {y.shape}')    # (5, 3, 2, 4)

inverse_dims = tuple(int(i) for i in np.argsort(dims))
print(f'dims: {dims}')                # (3, 1, 0, 2)
print(f'inverse: {inverse_dims}')    # (2, 1, 3, 0)

grad_out = t.randn_like(y)
grad_x = permute_back(grad_out, y, x, dims)
print(f'grad_x shape: {grad_x.shape}')  # (2, 3, 4, 5)
print(f'Shape matches x: {grad_x.shape == x.shape}')   # True
print(f'Round-trip identity: {t.equal(y.permute(*inverse_dims), x)}')  # True